In [1]:
#https://platform.openai.com/docs/guides/tools-code-interpreter

In [2]:
from dotenv import load_dotenv
_ = load_dotenv()

import json
import openai
from openai import OpenAI

client = OpenAI()

In [3]:
from sklearn.datasets import load_iris
iris = load_iris()

In [4]:
import pandas as pd
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df['species'] = iris.target
df['species_name'] = df['species'].apply(lambda x: iris.target_names[x])
df = df.drop('species', axis=1)
df = df.rename(columns={'species_name': 'species'})
df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [5]:
df.to_csv('iris.csv', index=False)

In [6]:
# ----------------------------
# 2. Upload file
# ----------------------------
file_obj = client.files.create(file=open("iris.csv", "rb"), purpose="user_data")
file_id = file_obj.id
print(f"Uploaded file ID: {file_id}")

Uploaded file ID: file-9g5yd2SYico995tVtGZtyv


In [7]:

instructions = """
You are a biologist and statistician.
You have collection of sepal and petal dimensions for 150 iris flowers.
The csv contains 1 header row, no index column, 4 measurement columns and a species column.
Select 80% of the data rows to learn and predict the species of the other 20%.
Produce an ASCII style confusion matrix of the prediction results.
"""

# by default it actually produced a PNG confusion matrix but 
# I can't quite figure out how to extract an image produced here.

#This is a perishable container that I will not have access to after this query.
#Save the confusion matrix as a globally accessible File and return the file_id so that I may download it later.

#Emit the entire confustion matrix PNG as base64 in its own attribute in the response so that I can then print in Python

#          "text": "Here is your confusion matrix as a base64-encoded PNG image. You can decode and display it in Python using standard 
#libraries such as matplotlib or PIL.\n\n```json\n{\n  \"confusion_matrix_png_base64\": \"iVBORw0KGgoAAAANSUhEUgAABLAAAASwCAYAAADrIbPPAAAAOXRFWHRTb2Z0d2FyZQBNYXRwbG90bGliIHZlcnNpb24zLjYuMywg...<truncated>\"\n}\n```\n
#\nReplace `...<truncated>` with the full string below for use in your Python environment.\n\nLet me know if you'd like the full base64 string or the code to decode and print it!",


In [8]:
resp = client.responses.create(
    model="gpt-4.1",
    tools=[
        {
            "type": "code_interpreter",
            "container": {
                "type": "auto",
                "file_ids": [file_id]
            }
        }
    ],
    input=[
        {"role": "system", "content": instructions}],
#        {"role": "user", "content": q}],


)


In [9]:
print(resp.output)

[ResponseCodeInterpreterToolCall(id='ci_02fe796983239d450068fc26ffa5fc81a08336e7842f455010', code='import pandas as pd\n\n# Load the dataset\nfile_path = "/mnt/data/file-9g5yd2SYico995tVtGZtyv-iris.csv"\niris_df = pd.read_csv(file_path)\n\n# Show the first few rows to understand the format\niris_df.head()', container_id='cntr_68fc26fe1a188191aa6572ae4eb99e3702a096b197dce64a', outputs=None, status='completed', type='code_interpreter_call'), ResponseCodeInterpreterToolCall(id='ci_02fe796983239d450068fc2709709081a0ba76d7007cb94018', code='from sklearn.model_selection import train_test_split\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import confusion_matrix\nimport numpy as np\n\n# Prepare data\nX = iris_df.iloc[:, :-1]\ny = iris_df[\'species\']\n\n# Split the dataset into train (80%) and test (20%)\nX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)\n\n# Train a classifier (Random Forest for robustness)\ncl

In [10]:
pretty_json_output = resp.model_dump_json(indent=2)
print(pretty_json_output)

{
  "id": "resp_02fe796983239d450068fc26fc294881a0b881e621a526cfca",
  "created_at": 1761355516.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-4.1-2025-04-14",
  "object": "response",
  "output": [
    {
      "id": "ci_02fe796983239d450068fc26ffa5fc81a08336e7842f455010",
      "code": "import pandas as pd\n\n# Load the dataset\nfile_path = \"/mnt/data/file-9g5yd2SYico995tVtGZtyv-iris.csv\"\niris_df = pd.read_csv(file_path)\n\n# Show the first few rows to understand the format\niris_df.head()",
      "container_id": "cntr_68fc26fe1a188191aa6572ae4eb99e3702a096b197dce64a",
      "outputs": null,
      "status": "completed",
      "type": "code_interpreter_call"
    },
    {
      "id": "ci_02fe796983239d450068fc2709709081a0ba76d7007cb94018",
      "code": "from sklearn.model_selection import train_test_split\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import confusion_matrix\nimport numpy as np\n

In [11]:
data = json.loads(resp.model_dump_json(include={"output"}))

In [12]:
#data['output']

In [13]:

# Loop through the list of dictionaries
for item in data['output']:
    # Each 'item' in this loop is a dictionary
#    print(f"ID: {item['id']}, Name: {item['name']}, City: {item['city']}")
    if 'code' in item:
        print(item['code'])
    if 'content' in item:
        print(item['content'][0]['text'])

import pandas as pd

# Load the dataset
file_path = "/mnt/data/file-9g5yd2SYico995tVtGZtyv-iris.csv"
iris_df = pd.read_csv(file_path)

# Show the first few rows to understand the format
iris_df.head()
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
import numpy as np

# Prepare data
X = iris_df.iloc[:, :-1]
y = iris_df['species']

# Split the dataset into train (80%) and test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train a classifier (Random Forest for robustness)
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)

# Predict species for test set
y_pred = clf.predict(X_test)

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)
species = clf.classes_

# Prepare ASCII confusion matrix
output = "Confusion Matrix:\n\n"
output += " " * 15 + "".join([f"{s:>12}" for s i

In [14]:
#display(Image("your_image.png")) 


#import base64, io
#from IPython.display import Image, display
#
#b64 = "iVBORw0K..."  # string from response
#png_bytes = base64.b64decode(b64)
#display(Image(data=png_bytes))
